In [1]:
import dspy
lm = dspy.LM('ollama_chat/gemma3:4b', api_base='http://localhost:11434', api_key='')
dspy.configure(lm=lm)

/home/penhfel/miniconda3/envs/chatbot-study/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fine Tuning for Classification (Stance)

In [2]:
import json
with open("../../training-conversational-stance/Tests/SemEval 2019/ModernBert/supervised_training_data.json") as file_handler:
    train_data = json.load(file_handler)

In [3]:
with open('../../training-conversational-stance/Tests/SemEval 2019/ModernBert/supervised_test_data.json') as file_handler:
    test_data = json.load(file_handler)

In [4]:
CLASSES = list(set([train_data[key]['label'] for key in train_data]))

In [5]:
trainset = [
    dspy.Example(message=train_data[key]['conversation'][-1]['message'], hint=train_data[key]['label'], answer=train_data[key]['label']).with_inputs("message", "hint")
    for key in train_data
]

In [6]:
trainset[0]

Example({'message': 'France: 10 people dead after shooting at HQ of satirical weekly newspaper #CharlieHebdo, according to witnesses http://t.co/FkYxGmuS58', 'hint': 'support', 'answer': 'support'}) (input_keys={'message', 'hint'})

In [7]:
from typing import Literal

signature = dspy.Signature("message -> answer").with_updated_fields('answer', type_=Literal[tuple(CLASSES)])
classify = dspy.ChainOfThought(signature)

In [8]:
classify

predict = Predict(StringSignature(message -> reasoning, answer
    instructions='Given the fields `message`, produce the fields `answer`.'
    message = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Message:', 'desc': '${message}'})
    reasoning = Field(annotation=str required=True json_schema_extra={'prefix': "Reasoning: Let's think step by step in order to", 'desc': '${reasoning}', '__dspy_field_type': 'output'})
    answer = Field(annotation=Literal['query', 'deny', 'comment', 'support'] required=True json_schema_extra={'__dspy_field_type': 'output', 'prefix': 'Answer:', 'desc': '${answer}'})
))

In [9]:
from typing import Literal

class Classify(dspy.Signature):
    """You are an evaluator who needs to evaluate a conversation about a news in Twitter.
This news needs to be checked whether it is a fake news or not. Your job is to decide if the current tweet support, deny, query or add a comment about the subject of the news. You need to only complete with one of the following stance: support, deny, query or comment.
When an message support that the news is true, they NEED to be classified as support. This can be show by it's own messages or when the user is in favor of an user that supports the news.
When an message questions the validity of the news, consider it a lie or asks for other users to refute it, they NEED to be classified as deny. This can be show by it's own messages or when the user is in favor of an user that denies the news.
When an message is asking about new information regarding the news, they NEED to be classified as query.
When the message does not adds more information about the news or is not in favor or against other user, they NEED to be classified as comment.
"""

    message: str = dspy.InputField()
    answer: Literal[tuple(CLASSES)] = dspy.OutputField()
    confidence: float = dspy.OutputField()

classify = dspy.ChainOfThought(Classify)

In [10]:
trainset[0]

Example({'message': 'France: 10 people dead after shooting at HQ of satirical weekly newspaper #CharlieHebdo, according to witnesses http://t.co/FkYxGmuS58', 'hint': 'support', 'answer': 'support'}) (input_keys={'message', 'hint'})

In [11]:
classify(message= trainset[0].message)

Prediction(
    reasoning="The tweet reports a shooting at the headquarters of Charlie Hebdo, a satirical weekly newspaper. This is a significant news event. The tweet provides a link to a source, suggesting it's a factual report. Therefore, it warrants support for the news.",
    answer='support',
    confidence=1.0
)

In [12]:
tp = dspy.MIPROv2(metric=dspy.evaluate.answer_exact_match, auto="light", num_threads=1, verbose=True)
optimized_react = tp.compile(classify, trainset=trainset)

2025/03/22 20:26:46 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 7
minibatch: True
num_candidates: 5
valset size: 100

2025/03/22 20:26:48 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2025/03/22 20:26:48 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2025/03/22 20:26:48 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=5 sets of demonstrations...


Bootstrapping set 1/5
Bootstrapping set 2/5
Bootstrapping set 3/5


  0%|          | 12/5702 [00:25<3:24:01,  2.15s/it]


Bootstrapped 4 full traces after 12 examples for up to 1 rounds, amounting to 12 attempts.
Bootstrapping set 4/5


  0%|          | 5/5702 [00:08<2:36:03,  1.64s/it]


...

2025/03/22 20:33:52 INFO dspy.evaluate.evaluate: Average Metric: 13 / 25 (52.0%)
2025/03/22 20:33:52 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 52.0 on minibatch of size 25 with parameters ['Predictor 0: Instruction 4', 'Predictor 0: Few-Shot Set 3'].
2025/03/22 20:33:52 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [88.0, 84.0, 80.0, 80.0, 52.0]
2025/03/22 20:33:52 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [44.0]
2025/03/22 20:33:52 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 44.0
2025/03/22 20:33:52 INFO dspy.teleprompt.mipro_optimizer_v2: ========================================


2025/03/22 20:33:52 INFO dspy.teleprompt.mipro_optimizer_v2: == Trial 7 / 8 - Minibatch ==
2025/03/22 20:33:52 INFO dspy.teleprompt.mipro_optimizer_v2: Evaluating the following candidate program...




Predictor 0
i: You are an evaluator who needs to evaluate a conversation about a news in Twitter.
This news needs to be checked whether it is a fake news or not. Your job is to decide if the current tweet support, deny, query or add a comment about the subject of the news. You need to only complete with one of the following stance: support, deny, query or comment.
When an message support that the news is true, they NEED to be classified as support. This can be show by it's own messages or when the user is in favor of an user that supports the news.
When an message questions the validity of the news, consider it a lie or asks for other users to refute it, they NEED to be classified as deny. This can be show by it's own messages or when the user is in favor of an user that denies the news.
When an message is asking about new information regarding the news, they NEED to be classified as query.
When the message does not adds more information about the news or is not in favor or against ot

2025/03/22 20:34:39 INFO dspy.evaluate.evaluate: Average Metric: 21 / 25 (84.0%)
2025/03/22 20:34:39 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 84.0 on minibatch of size 25 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 1'].
2025/03/22 20:34:39 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [88.0, 84.0, 80.0, 80.0, 52.0, 84.0]
2025/03/22 20:34:39 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [44.0]
2025/03/22 20:34:39 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 44.0
2025/03/22 20:34:39 INFO dspy.teleprompt.mipro_optimizer_v2: ========================================


2025/03/22 20:34:39 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 8 / 8 - Full Evaluation =====
2025/03/22 20:34:39 INFO dspy.teleprompt.mipro_optimizer_v2: Doing full eval on next top averaging program (Avg Score: 88.0) from minibatch trials...



Average Metric: 83.00 / 100 (83.0%): 100%|██████████| 100/100 [02:28<00:00,  1.49s/it]

2025/03/22 20:37:08 INFO dspy.evaluate.evaluate: Average Metric: 83 / 100 (83.0%)
2025/03/22 20:37:08 INFO dspy.teleprompt.mipro_optimizer_v2: New best full eval score! Score: 83.0
2025/03/22 20:37:08 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [44.0, 83.0]
2025/03/22 20:37:08 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 83.0
2025/03/22 20:37:08 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2025/03/22 20:37:08 INFO dspy.teleprompt.mipro_optimizer_v2: 

2025/03/22 20:37:08 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 83.0!


In [35]:
optimized_react.predict.signature

StringSignature(message -> reasoning, answer, confidence
    instructions='You are a news analyst specializing in quickly assessing the sentiment and veracity of Twitter posts related to breaking news events. Your task is to analyze a user-provided tweet and determine whether it supports, denies, queries, or adds a comment regarding the news. Respond with only one of the following classifications: "support", "deny", "query", or "comment". Provide a brief reasoning for your classification, and assign a confidence score (between 0 and 1) reflecting your certainty.'
    message = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Message:', 'desc': '${message}'})
    reasoning = Field(annotation=str required=True json_schema_extra={'prefix': "Reasoning: Let's think step by step in order to", 'desc': '${reasoning}', '__dspy_field_type': 'output'})
    answer = Field(annotation=Literal['query', 'deny', 'comment', 'support'] required=True json_sche

In [14]:
testset = [
    dspy.Example(message=test_data[key]['conversation'][-1]['message'], hint=test_data[key]['label'], answer=test_data[key]['label']).with_inputs("message", "hint")
    for key in test_data
]

In [15]:
classify(message= testset[0].message)

Prediction(
    reasoning="The user is presenting a personal anecdote about a potential link between red dye #40 and their child's ADHD symptoms. While the user is seeking verification and has a strong belief based on their friend's experience, there isn't definitive scientific evidence to support a direct causal link between red dye #40 and ADHD. The user's reaction – a ‘good hard stare’ – suggests skepticism, but the core of the message is a question seeking validation. Therefore, classifying this as a query is appropriate.",
    answer='query',
    confidence=0.85
)

In [16]:
optimized_react(message= testset[0].message)

Prediction(
    reasoning="The user is presenting a personal anecdote and a claim about a potential link between red dye #40 and ADHD symptoms. While there have been past concerns and studies regarding food dyes and hyperactivity in children, the scientific consensus is that there is no definitive evidence to support a causal link. The user is seeking information to evaluate the validity of their friend's claim.",
    answer='comment',
    confidence=0.65
)

In [30]:
from sklearn.metrics import classification_report

In [ ]:
# from tqdm import tqdm
# predictions = []
# y_true = []
# for example in tqdm(testset):
#     prediction = classify(message= example.message)
#     predictions.append(prediction)
#     y_true.append(example.answer)

# y_pred = [x.answer for x in predictions]

# for i, key in enumerate(test_data):
#     test_data[key]['prediction'] = y_pred[i]

# with open('../../training-conversational-stance/Predictions/SemEval 2019/gemma_4b_dspy.json', 'w') as file_handler:
#     json.dump(test_data,file_handler)

100%|██████████| 1827/1827 [44:32<00:00,  1.46s/it] 


Optimized Prompt

In [24]:
from tqdm import tqdm
predictions_optimized = []
y_true_optimized = []
for example in tqdm(testset):
    prediction = optimized_react(message= example.message)
    predictions_optimized.append(prediction)
    y_true_optimized.append(example.answer)

 18%|█▊        | 325/1827 [1:20:06<6:10:12, 14.79s/it] 


KeyboardInterrupt: 

In [27]:
y_pred_optimized = [x.answer for x in predictions_optimized]

In [28]:
print(classification_report(y_true=y_true_optimized,y_pred=y_pred_optimized))

              precision    recall  f1-score   support

     comment       0.93      0.91      0.92       290
        deny       0.00      0.00      0.00         6
       query       0.32      0.42      0.36        19
     support       0.00      0.00      0.00        10

    accuracy                           0.83       325
   macro avg       0.31      0.33      0.32       325
weighted avg       0.85      0.83      0.84       325

